# Análise Exploratória das Tribos Literárias (Data Storytelling)

Neste notebook, faremos uma jornada exploratória para desvendar a personalidade e os hábitos de leitura das **3 Tribos Literárias** que mapeamos em nosso catálogo de livros do Booklog. 

Em vez de olharmos apenas para números isolados, buscaremos contar a história de quem são essas pessoas: o que elas leem, como elas avaliam suas obras, se preferem livros rápidos ou calhamaços densos, e como essas características as agrupam em "galáxias" distintas de comportamento.

### As Nossas Três Personagens (As Tribos):
* **Tribo 0: "Os Especialistas & Autodesenvolvimento"** (Buscam aprendizado prático, estilo de vida e habilidades técnicas. Leitores focados).
* **Tribo 1: "Os Acadêmicos & Clássicos"** (Devoradores de biografia, história e grandes clássicos literários. Leituras densas e maduras).
* **Tribo 2: "A Ficção Pop & Mainstream"** (A grande engrenagem de engajamento do Booklog. Fãs de romance, fantasia, ficção adolescente e best-sellers populares).


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import kruskal

print("Bibliotecas importadas com sucesso!")


Bibliotecas importadas com sucesso!


## 1. Entrando na Biblioteca: Carregamento do Catálogo com Clusters
Carregamos os dados contendo a designação de cada livro para a sua respectiva Tribo Literária (Cluster).

> **Nota sobre o número de colunas (De 10 para 17 colunas):**
> O dataset original (`books.parquet`) possuía 10 colunas brutas. No entanto, o nosso dataset processado final agora conta com **17 colunas**. Por que isso aconteceu?
> 1. **Engenharia de Gêneros Pivotados (9 colunas):** Em vez de mantermos uma única coluna com listas de gêneros em formato texto, transformamos os gêneros consolidados em **9 colunas binárias** (0 ou 1) independentes.
> 2. **Variáveis Numéricas Originais (3 colunas):** Mantivemos `rating`, `pages` e `totalratings`.
> 3. **Identidade e Metadados do Modelo (5 colunas):** Temos as colunas `title` e `author` (identificação), a coluna `Cluster` (grupo classificado pelo K-Means) e as coordenadas de projeção visual `svd_x` e `svd_y`.
>
> Essa transformação (12 variáveis de modelagem + 5 de suporte) é o que viabiliza a interpretabilidade dos clusters e a plotagem do dashboard.


In [2]:
# Carregar os dados de livros com clusters
df = pd.read_parquet('Machine Learning/data/processed/livros_com_clusters.parquet')
print(f"Catálogo carregado com {df.shape[0]:,} livros e {df.shape[1]} colunas.")
print("\nExemplo dos primeiros livros com suas colunas e coordenadas:")
print(df[['title', 'author', 'Cluster', 'svd_x', 'svd_y']].head(3))


Catálogo carregado com 81,979 livros e 17 colunas.
Exemplo dos primeiros livros com suas colunas e coordenadas:
                                    title  ...     svd_y
0                              "Daisuki."  ... -0.127454
1       "Dark Pictures" and Other Stories  ...  0.222201
2  "Defects": Engendering the Modern Body  ...  0.167648
[3 rows x 5 columns]


## 2. Capítulo 1: O Tamanho das Tribos
A primeira pergunta da nossa história é: **como o catálogo do Booklog está distribuído?** Será que temos uma tribo gigantesca e outras minúsculas, ou as três dividem o catálogo de forma equilibrada?

Plotamos o volume de livros em cada tribo para entender essa divisão.


In [3]:
# Contagem e percentual por cluster
dist_df = df['Cluster'].value_counts().reset_index()
dist_df.columns = ['Tribo', 'Quantidade']
dist_df['Percentual'] = (dist_df['Quantidade'] / dist_df['Quantidade'].sum()) * 100
dist_df['Nome_Tribo'] = dist_df['Tribo'].map({
    0: 'Tribo 0: Não-Ficção & Autodesenvolvimento',
    1: 'Tribo 1: Clássicos, Biografias & Literatura',
    2: 'Tribo 2: Ficção Pop, Fantasia & Romance'
})

# Mapear cores estritas solicitadas: Tribo 0 = Vermelho, Tribo 1 = Azul, Tribo 2 = Verde
CORES_MAP = {
    'Tribo 0: Não-Ficção & Autodesenvolvimento': '#e41a1c', # Vermelho
    'Tribo 1: Clássicos, Biografias & Literatura': '#377eb8', # Azul
    'Tribo 2: Ficção Pop, Fantasia & Romance': '#4daf4a'  # Verde
}

# Criar gráfico de barras
fig_dist = px.bar(
    dist_df,
    x='Nome_Tribo',
    y='Quantidade',
    text=dist_df['Percentual'].apply(lambda x: f"{x:.1f}%"),
    title='Distribuição dos Livros do Catálogo pelas 3 Tribos Literárias',
    color='Nome_Tribo',
    color_discrete_map=CORES_MAP,
    labels={'Quantidade': 'Número de Livros', 'Nome_Tribo': 'Tribo Literária'}
)
fig_dist.update_traces(textposition='outside')
fig_dist.update_layout(height=450, showlegend=False, margin=dict(t=50, b=20, l=20, r=20))
fig_dist.show()


* **Insight do Storytelling:** Vemos que o catálogo é muito bem equilibrado! 
  * A **Tribo 0 (Não-Ficção de Nicho)** responde pela maior fatia (38.5%), sugerindo um vasto acervo de manuais, guias e livros práticos.
  * A **Tribo 2 (Ficção Pop e Romance)** vem logo em seguida com 35.4%, representando a alma jovem e engajada da plataforma.
  * A **Tribo 1 (Clássicos e Biografias)** ocupa 26.2% do acervo, mostrando uma fatia sólida de literatura consagrada e informativa.

---

## 3. Capítulo 2: A Preferência Literária das Tribos
Agora, vamos investigar o **gosto literário de cada grupo**. O que define a identidade de leitura dessas tribos? 

Para isso, calculamos a taxa de presença de cada um dos 9 gêneros principais dentro de cada cluster, nos dando o "DNA" de cada grupo.

> **Importante: Por que a soma das barras de uma mesma tribo ultrapassa 100%?**
> Ao olhar para o gráfico a seguir, você notará que a soma das porcentagens dos gêneros de um mesmo cluster ultrapassa 100%. **Isso não é um erro!**
> Isso acontece porque **um mesmo livro pode ter múltiplos gêneros marcados simultaneamente (dados multilabel)**. Por exemplo, uma mesma obra pode ser marcada como "Romance" e também "Ficção Geral e Literatura".
> A barra no gráfico não representa uma divisão exclusiva do catálogo, mas sim: *"Qual a porcentagem de livros desta tribo que possui este gênero marcado?"*. Como os livros se sobrepõem entre os gêneros, as categorias não são excludentes, fazendo com que a soma natural dos percentuais no cluster seja maior do que 100%.


In [4]:
genre_cols = [
    'Artes, Lazer e Estilo de Vida',
    'Fantasia e Ficção Científica',
    'Ficção Geral e Literatura',
    'História e Biografia',
    'Infantojuvenil e Quadrinhos',
    'Mistério, Thriller e Terror',
    'Não-Ficção e Autodesenvolvimento',
    'Outros',
    'Romance'
]

# Calcular a proporção de cada gênero por cluster
proporcoes = df.groupby('Cluster')[genre_cols].mean().reset_index()

# Pivotar dados para o formato longo (long format) para exibição no Plotly
prop_melted = proporcoes.melt(id_vars='Cluster', var_name='Gênero', value_name='Proporção')
prop_melted['Percentual'] = prop_melted['Proporção'] * 100
prop_melted['Nome_Tribo'] = prop_melted['Cluster'].map({
    0: 'Tribo 0 (Não-Ficção & Desenvolvimento)',
    1: 'Tribo 1 (Clássicos, Biografia & Literatura)',
    2: 'Tribo 2 (Ficção Pop, Fantasia & Romance)'
})

# Mapear cores estritas solicitadas: Tribo 0 = Vermelho, Tribo 1 = Azul, Tribo 2 = Verde
CORES_GENRES = {
    'Tribo 0 (Não-Ficção & Desenvolvimento)': '#e41a1c', # Vermelho
    'Tribo 1 (Clássicos, Biografia & Literatura)': '#377eb8', # Azul
    'Tribo 2 (Ficção Pop, Fantasia & Romance)': '#4daf4a'  # Verde
}

# Criar gráfico de barras agrupado por Tribo
fig_genres = px.bar(
    prop_melted,
    x='Percentual',
    y='Gênero',
    color='Nome_Tribo',
    barmode='group',
    title='Presença de Gêneros Literários por Tribo',
    color_discrete_map=CORES_GENRES,
    labels={'Percentual': 'Presença no Cluster (%)', 'Gênero': 'Gênero Literário', 'Nome_Tribo': 'Tribo'}
)
fig_genres.update_layout(height=650, yaxis={'categoryorder':'total ascending'}, margin=dict(t=50, b=20, l=20, r=20))
fig_genres.show()


* **Análise Detalhada do DNA Literário (Storytelling):**
  * **Tribo 0 (Vermelho):** O foco principal é pragmático e voltado para aprendizado ou hobbies. É a tribo da *Não-Ficção e Autodesenvolvimento* (82% de presença) e *Artes, Lazer e Estilo de Vida* (37%). 
  * **Tribo 1 (Azul):** A tribo da leitura densa e clássica. Caracteriza-se por *Ficção Geral e Literatura* (presente em 100% dos livros da tribo) associada fortemente a *História e Biografia* (55% de presença). É o grupo dos calhamaços de história e romances clássicos consagrados.
  * **Tribo 2 (Verde):** A tribo da ficção popular e entretenimento. Aqui vemos a sobreposição dos gêneros multilabel: **87%** dos seus livros possuem a tag de *Ficção Geral*, enquanto **48%** possuem *Fantasia e Ficção Científica* e **47%** contam com *Romance*. Muitos livros desse grupo carregam mais de uma dessas tags ao mesmo tempo (como um romance fantástico ou uma ficção infantojuvenil).

Essa análise revela que a Tribo 2 consome os livros de maior apelo de entretenimento e comercial (Fantasia + Romance), a Tribo 0 foca em desenvolvimento prático e a Tribo 1 concentra leituras literárias clássicas e históricas.

---

## 4. Capítulo 3: Os Hábitos de Consumo e Avaliação
Sabemos o que eles leem, mas **como** eles consomem esses livros?
* Eles leem calhamaços ou livros de leitura rápida?
* Suas tribos são rigorosas ou generosas nas notas médias?
* Suas leituras são fenômenos mundiais (milhões de avaliações) ou obras de nicho?

Analisamos isso através de boxplots comparativos para a Nota Média (`rating`), Extensão (`pages`) e Popularidade (`totalratings`).


In [5]:
fig_box = make_subplots(
    rows=1, cols=3, 
    subplot_titles=('Nota Média (Rating)', 'Extensão (Páginas)', 'Popularidade (Avaliações)')
)

# Copiar dataframe e mapear nomes legíveis
df_plot = df.copy()
df_plot['Nome_Tribo'] = df_plot['Cluster'].map({
    0: 'Tribo 0 (Não-Ficção)',
    1: 'Tribo 1 (Clássicos)',
    2: 'Tribo 2 (Ficção Pop)'
})

# Adicionar Boxplots individuais para cada cluster com cores estritas
for cluster_id, color in zip([0, 1, 2], ['#e41a1c', '#377eb8', '#4daf4a']):
    subset = df_plot[df_plot['Cluster'] == cluster_id]
    
    fig_box.add_trace(
        go.Box(y=subset['rating'], name=f"Tribo {cluster_id}", marker_color=color, showlegend=False),
        row=1, col=1
    )
    fig_box.add_trace(
        go.Box(y=subset['pages'], name=f"Tribo {cluster_id}", marker_color=color, showlegend=False),
        row=1, col=2
    )
    fig_box.add_trace(
        go.Box(y=subset['totalratings'], name=f"Tribo {cluster_id}", marker_color=color, showlegend=False),
        row=1, col=3
    )

# Definir limites de zoom para podermos ler a caixa central ignorando os outliers gigantescos
fig_box.update_yaxes(range=[2.5, 5.0], row=1, col=1)
fig_box.update_yaxes(range=[0, 650], row=1, col=2)
fig_box.update_yaxes(range=[0, 15000], row=1, col=3)

fig_box.update_layout(
    title_text='Distribuição das Características Físicas e de Popularidade por Tribo',
    height=480,
    margin=dict(t=80, b=20, l=20, r=20)
)
fig_box.show()


* **Insights de Storytelling dos Hábitos:**
  * **As Notas:** A **Tribo 0 (Não-Ficção)** e a **Tribo 1 (Clássicos)** têm avaliações médias muito altas (~3.93 e 3.91). Isso mostra que leitores de não-ficção e clássicos tendem a dar boas notas, ou que estas obras passam por seleções de curadoria rigorosas. A **Tribo 2 (Ficção Pop)** tem a menor nota mediana (3.84), mostrando que o público pop/jovem é altamente engajado, mas também mais crítico.
  * **O Tamanho:** A **Tribo 1 (Clássicos)** lê os livros mais longos (mediana próxima a 287 páginas). A **Tribo 2 (Ficção Pop)** lê os livros mais curtos (mediana de 239 páginas), refletindo leituras rápidas e dinâmicas (como mangás, quadrinhos e romances rápidos).
  * **A Popularidade (Avaliações):** Aqui está a maior discrepância do dataset! O Boxplot da direita mostra que a **Tribo 2 (Ficção Pop)** tem um volume de avaliações estrondoso (mediana de 6.165 e média puxada para cima por best-sellers). A **Tribo 0 (Não-Ficção)** tem uma popularidade muito baixa (média de 615 avaliações), evidenciando livros técnicos e guias de nicho.

---

## 5. Capítulo 4: Validação Científica da Diferença de Perfis
Para provar que essas diferenças de comportamento (páginas, avaliações e notas) não aconteceram por acaso, aplicamos o teste estatístico **Kruskal-Wallis**. Este teste analisa se as distribuições das 3 tribos são realmente diferentes de forma estatisticamente significativa.


In [6]:
print("=== Validação Estatística de Diferenças entre Grupos (Kruskal-Wallis) ===")
print("Hipótese Nula (H0): Não há diferença nas distribuições das variáveis entre os clusters.")
print("Hipótese Alternativa (H1): Pelo menos um cluster apresenta distribuição diferente.")
print("-" * 75)

for col in ['rating', 'pages', 'totalratings']:
    g0 = df[df['Cluster'] == 0][col]
    g1 = df[df['Cluster'] == 1][col]
    g2 = df[df['Cluster'] == 2][col]
    
    stat, p_val = kruskal(g0, g1, g2)
    print(f"Variável: {col:<12} | Estatística H: {stat:10.2f} | p-value: {p_val:.2e}")
    if p_val < 0.05:
        print(f"  -> Conclusão: Rejeitamos H0. A diferença entre as tribos é ESTATISTICAMENTE SIGNIFICATIVA (p < 0.05).")
    else:
        print(f"  -> Conclusão: Aceitamos H0. Não há diferença estatisticamente significativa.")
    print("-" * 75)


=== Validação Estatística de Diferenças entre Grupos (Kruskal-Wallis) ===
Hipótese Nula (H0): Não há diferença nas distribuições das variáveis entre os clusters.
Hipótese Alternativa (H1): Pelo menos um cluster apresenta distribuição diferente.
---------------------------------------------------------------------------
Variável: rating       | Estatística H:    1504.70 | p-value: 0.00e+00
  -> Conclusão: Rejeitamos H0. A diferença entre as tribos é ESTATISTICAMENTE SIGNIFICATIVA (p < 0.05).
---------------------------------------------------------------------------
Variável: pages        | Estatística H:     892.25 | p-value: 1.78e-194
  -> Conclusão: Rejeitamos H0. A diferença entre as tribos é ESTATISTICAMENTE SIGNIFICATIVA (p < 0.05).
---------------------------------------------------------------------------
Variável: totalratings | Estatística H:   20089.36 | p-value: 0.00e+00
  -> Conclusão: Rejeitamos H0. A diferença entre as tribos é ESTATISTICAMENTE SIGNIFICATIVA (p < 0.05).
-

* **Insight Acadêmico:** Com um p-value de praticamente zero ($p < 0.001$), rejeitamos a hipótese nula com segurança máxima. Isso prova que o algoritmo K-Means separou o catálogo em grupos que possuem hábitos de leitura **inerentemente diferentes** em termos de nota, tamanho de livros e popularidade.

---

## 6. Capítulo 5: Mapeamento da Galáxia do Booklog (Mapa 2D do SVD)
Finalmente, vamos olhar para o **mapa espacial** dessas tribos. 
Usamos a projeção de **SVD (Singular Value Decomposition)** nas 12 variáveis de modelagem para plotar uma amostra de 10.000 livros em duas dimensões.

Cada ponto representa um livro, e a distância espacial reflete a semelhança entre eles.


In [7]:
# Amostra de 10.000 livros para performance visual no navegador
df_sample = df.sample(10000, random_state=42).copy()
df_sample['Cluster_Nome'] = df_sample['Cluster'].map({
    0: 'Tribo 0 (Não-Ficção & Autodesenvolvimento)',
    1: 'Tribo 1 (Clássicos, Biografias & Literatura)',
    2: 'Tribo 2 (Ficção Pop, Fantasia & Romance)'
})

# Mapear cores estritas solicitadas: Tribo 0 = Vermelho, Tribo 1 = Azul, Tribo 2 = Verde
CORES_SCATTER = {
    'Tribo 0 (Não-Ficção & Autodesenvolvimento)': '#e41a1c', # Vermelho
    'Tribo 1 (Clássicos, Biografias & Literatura)': '#377eb8', # Azul
    'Tribo 2 (Ficção Pop, Fantasia & Romance)': '#4daf4a'  # Verde
}

# Gerar gráfico de dispersão 2D com as coordenadas SVD originais
fig_scatter = px.scatter(
    df_sample,
    x='svd_x',
    y='svd_y',
    color='Cluster_Nome',
    hover_name='title',
    hover_data=['author', 'rating', 'pages'],
    title='Mapa Espacial da Galáxia de Livros (Amostra de 10.000 títulos via SVD)',
    category_orders={'Cluster_Nome': [
        'Tribo 0 (Não-Ficção & Autodesenvolvimento)',
        'Tribo 1 (Clássicos, Biografias & Literatura)',
        'Tribo 2 (Ficção Pop, Fantasia & Romance)'
    ]},
    color_discrete_map=CORES_SCATTER,
    labels={'svd_x': 'Componente de Projeção SVD 1', 'svd_y': 'Componente de Projeção SVD 2'}
)

fig_scatter.update_traces(marker=dict(size=4, opacity=0.6, line=dict(width=0.2, color='DarkSlateGrey')))
fig_scatter.update_layout(
    height=600,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(t=80, b=20, l=20, r=20)
)
fig_scatter.show()


## Conclusão da Jornada: Resumo das Tribos

Ao fim do nosso Storytelling, podemos categorizar as personas de leitura do Booklog com precisão cirúrgica:

1. **Os Especialistas (Tribo 0):** Representam 38% do catálogo. Leem livros de tamanho moderado sobre negócios, estilo de vida e aprendizado prático. Suas leituras são altamente direcionadas (nicho de mercado, poucas avaliações gerais), mas extremamente apreciadas (altas avaliações).
2. **Os Acadêmicos (Tribo 1):** Representam 26% do catálogo. Devoram calhamaços históricos, biografias densas e literatura de prestígio. São leitores focados em conhecimento profundo e qualidade.
3. **O Motor Pop (Tribo 2):** Representa 35% do catálogo. Devoram romances e mundos fantásticos. Embora os livros sejam mais curtos e as notas ligeiramente mais dispersas, essa tribo concentra o **maior barulho e engajamento da comunidade**, com um volume massivo de resenhas e compartilhamentos.

Essas informações são valiosas para orientar campanhas de marketing direcionadas, sugerir novas aquisições de catálogo e desenhar o sistema de recomendação do Booklog!
